In [ ]:
import pandas as pd
from jarvis.db.figshare import data
from matminer.featurizers.composition import ElementProperty
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt


In [ ]:
mxene_data = data("mxene275")
df = pd.DataFrame(mxene_data)

# make column formula from id
df["formula"] = df["id"].str.replace("POSCAR-", "").str.replace(".vasp", "")

print(df[["id", "formula", "formation_energy"]].head())


In [ ]:
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

# make column composition from formula
df["composition"] = df["formula"].apply(lambda x: Composition(x))

# magpie
ep = ElementProperty.from_preset(preset_name="magpie")
features = ep.featurize_dataframe(df, "composition", ignore_errors=True)

features.head()


In [ ]:
# only numerical column
X = features.select_dtypes(include=["number"]).drop(columns=["formation_energy"])
y = features["formation_energy"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))


In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("DFT Formation Energy (eV)")
plt.ylabel("Predicted Formation Energy (eV)")
plt.title("Parity Plot")
plt.show()


In [ ]:
import numpy as np

# importance feature 
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

# top 20
plt.figure(figsize=(10,6))
plt.title("Feature Importance (Top 20)")
plt.bar(range(20), importances[indices[:20]], align="center")
plt.xticks(range(20), X_train.columns[indices[:20]], rotation=90)
plt.ylabel("Importance")
plt.show()


In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score


groups = df["formula"].str.extract(r"([A-Z][a-z]?)")[0]

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(model, X, y, cv=gkf, groups=groups, scoring="r2")

print("R² per fold:", scores)
print("Mean R²:", scores.mean())


In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, scoring="r2", n_jobs=-1
)

plt.figure(figsize=(8,6))
plt.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Training score")
plt.plot(train_sizes, test_scores.mean(axis=1), "o-", label="Validation score")
plt.xlabel("Training examples")
plt.ylabel("R² score")
plt.title("Learning Curve - Random Forest")
plt.legend()
plt.show()


In [ ]:
import os

# make folder
os.makedirs("data/processed", exist_ok=True)

# save data predictions
pred_df.to_csv("data/processed/predictions.csv", index=False)


In [ ]:

pred_df = pd.DataFrame({
    "formula": df["formula"].iloc[y_test.index],  
    "formation_energy_true": y_test,
    "formation_energy_pred": y_pred
})

pred_df.to_csv("data/processed/predictions.csv", index=False)


In [ ]:
# all predictions
y_all_pred = model.predict(X)

pred_all_df = pd.DataFrame({
    "formula": df["formula"],
    "formation_energy_true": y,
    "formation_energy_pred": y_all_pred
})

import os
os.makedirs("data/processed", exist_ok=True)
pred_all_df.to_csv("data/processed/predictions_all.csv", index=False)


In [ ]:
import joblib
import os

os.makedirs("src", exist_ok=True) 
joblib.dump(model, "src/model_rf.joblib")


In [ ]:
def predict_formation_energy(formula_str):
    comp = Composition(formula_str)
    row = pd.DataFrame({"composition": [comp]})
    row_feat = ep.featurize_dataframe(row, "composition", ignore_errors=True)
    X_new = row_feat[model.feature_names_in_]
    return model.predict(X_new)[0]